#### Exkurs: Passwörter sicher in der Datenbank speichern:

Als erstes müssen wir natürlich eine Datenbank erstellen:

In [ ]:
import mysql.connector as mc


try:
    connection = mc.connect(host="localhost",
                            user="root",
                            passwd="12345",
                            use_pure=True)

except mc.errors.ProgrammingError:
    print("Es ist ein Fehler bei der Verbindung zur Datenbank aufgetreten!")

c = connection.cursor()

try:
    c.execute("CREATE DATABASE benutzer")
    print("Datenbank erstellt!")

except mc.Error as err:
    print("Es ist ein Fehler aufgetreten\n", err)

finally:
    connection.close()

#### Tabellen erstellen

Jetzt die benötigten Tabellen erstellen:

Als Länge für Passwort uns Salt nehmen wir die Passwortlänge des SHA-Algorithmus

In [ ]:
import mysql.connector as mc

try:
    connection = mc.connect(host="localhost",
                            user="root",
                            passwd="12345",
                            db="benutzer",
                            use_pure=True)

except mc.errors.ProgrammingError:
    print("Es ist ein Fehler bei der Verbindung zur Datenbank aufgetreten!")


c = connection.cursor()

sql = "CREATE TABLE benutzer (" \
      "id INT(6) UNSIGNED AUTO_INCREMENT PRIMARY KEY," \
      "benutzername VARCHAR(30) NOT NULL UNIQUE,"\
      "passwort VARCHAR (256) NOT NULL,"\
      "salt VARCHAR (256) NOT NULL"\
      ")"

c.execute(sql)

c.execute("SHOW TABLES")

for table in c:
    print(table)

connection.close()

#### Neuen Benutzer anlegen
Dabei wird ein Salt generiert, welches für die Verschlüsselung des Passworts benötigt wird. Dieses wird zusätzlich in der Datenbank gespeichert.

In [ ]:
import mysql.connector as mc
import os
import hashlib
# import mysql

try:
    connection = mc.connect(host="localhost",
                            user="root",
                            passwd="12345",
                            database="benutzer")

except mc.errors.ProgrammingError:
    print("Es ist ein Fehler bei der Verbindung zur Datenbank aufgetreten!")

c = connection.cursor()


name = "dirk1"
passwort = "123456"
salt = os.urandom(32)
print("Salt:", salt)
print("Salt:", salt.hex())


key = hashlib.pbkdf2_hmac(
        'sha512',  # Der Hash-Algorithmus, mehr ist besser
        passwort.encode('utf-8'),  # Konvertiere das Passwort in Bytes
        salt,  # Das Salt
        100  # Es wird empfohlen, mindestens 100000 Iterationen zu verwenden
    )

try:
    c.execute("""INSERT INTO benutzer (benutzername, passwort, salt) VALUES (%s,%s,%s)""", (name, key.hex(), salt.hex()))  # mit.hex() wird das Passort in einen "normalen" String umgewalt
    connection.commit()

except Exception as e:
    print(f"Fehler beim Ändern des Passwortes von: {name} Es ist der Fehler {e} aufgetreten.")

finally:
    connection.close()

#### login

Für einen erfolgreichen Login, wird das Passort und das Salt aus der Datenbank gelesen. Danach wird das **eingegebene** Passwort mit den gerade ausgelesenen Salt **verschlüsselt** und mit den gespeicherten **Hash** des gespeicherten Passworts verglichen.

Der Login ist erfolgreiche, wenn beide Hashes überein stimmen. 

In [ ]:
import mysql.connector as mc
import os
import hashlib

def verbindung_herstellen():
    try:
        connection = mc.connect(host="localhost",
                                user="root",
                                passwd="12345",
                                db="benutzer")
        c = connection.cursor()

        return connection, c

    except mc.errors.ProgrammingError:
        print("Es ist ein Fehler bei der Verbindung zur Datenbank aufgetreten!")


def login():
    name, passwort = eingabe()
    conn, c = verbindung_herstellen()
    sql = """SELECT passwort, salt FROM benutzer WHERE benutzername = %s"""
    params = (name, )
    c.execute(sql, params)
    result = c.fetchone()
    conn.close()

    if not result:
        print("Der Benutzer ist nicht vorhanden!")
        return False

    db_passwort = bytes.fromhex(result[0])  # Passwort aus der Datenbank in Bytes konvertiert
    db_salt = bytes.fromhex(result[1])  # Salt aus der Datenbank in Bytes konvertiert

    key = hashlib.pbkdf2_hmac("sha512",
    passwort.encode("utf-8"),
    db_salt, 100)  # Salt aus der Datenbank! Iteration muss mit Verschlüsselung überein stimmen

    if key == db_passwort:
        print("Login erfolgreich")
        return True
    else:
        print("Benutzername oder Passwort falsch")
        return False


def eingabe():
    name = input("Bitte geben Sie den Benutzernamen ein: ").lower()
    passwort = input("Bitte geben Sie das Passwort ein: ")

    return name, passwort


if login():
    print("Benutzer darf rein")
else:
    print("Benutzer darf nicht rein!")

### Sicheres Passwort

Das das eingegebene Passwort in der Konsole sichtbar ist, ist nicht gerade sicher. Dafür gibt es Bibliotheken, die das Passwort "verschleiern.

 #### Möglichkeit 1: getpass
 Das Passwort wird nicht ausgegeben, was das Erraten des Passworts erschwert. Die Bibliothek ist eine Standardbibliothek.

In [ ]:
from getpass import getpass

name = input("Benutzername: ")
passwort = getpass("Passwort: ")


if login(name, passwort):
    print("Erfolgreich eingeloggt!")
else:
    print("Du kommst hier nicht rein!!!")

 #### Möglichkeit 2: pwinput
 Das Passwort wird durch ein Zeichen (z. B. ein Sternchen) überschrieben. pwinput muss installiert
  werden

In [ ]:
from pwinput import pwinput  # muss installiert werden

name = input("Benutzername: ")

passwort = pwinput("Passwort: ", mask='*')  # Das Passwort wird bei Eingabe mit "*" angezeigt.


if login(name, passwort):
    print("Erfolgreich eingeloggt!")
else:
    print("Du kommst hier nicht rein!!!")

### Daten verschlüsseln & entschlüsseln (symmetrische Verschlüsselung)

In [ ]:
from cryptography.fernet import Fernet
import mysql.connector
from pathlib import Path

# ------------------------
# Schlüssel laden/erstellen
# ------------------------
KEY_FILE = Path("fernet.key")


def load_or_create_key():
    if KEY_FILE.exists():
        return KEY_FILE.read_bytes()
    else:
        key = Fernet.generate_key()
        KEY_FILE.write_bytes(key)
        return key


key = load_or_create_key()
cipher = Fernet(key)

# ------------------------
# MySQL-Verbindung
# ------------------------
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "12345",
    "database": "test_db",
    "use_pure": True

}


# ------------------------
# Funktion zum Speichern
# ------------------------
def save_secret_to_db(secret: str) -> int:
    """
    Verschlüsselt den Secret-Text und speichert ihn in der MySQL-Datenbank.
    Gibt die ID des gespeicherten Eintrags zurück.
    """
    encrypted = cipher.encrypt(secret.encode("utf-8"))

    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()

    cursor.execute("""
                   CREATE TABLE IF NOT EXISTS geheimdaten
                   (
                       id   INT AUTO_INCREMENT PRIMARY KEY,
                       info VARBINARY(500)
                   )
                   """)
    conn.commit()

    cursor.execute("INSERT INTO geheimdaten (info) VALUES (%s)", (encrypted,))
    conn.commit()

    last_id = cursor.lastrowid
    cursor.close()
    conn.close()

    return last_id


# ------------------------
# Funktion zum Abrufen
# ------------------------
def get_secret_from_db(entry_id: int) -> str:
    """
    Liest den verschlüsselten Text aus der DB anhand der ID und entschlüsselt ihn.
    """
    conn = mysql.connector.connect(**DB_CONFIG)
    cursor = conn.cursor()

    cursor.execute("SELECT info FROM geheimdaten WHERE id=%s", (entry_id,))
    encrypted = cursor.fetchone()[0]

    cursor.close()
    conn.close()

    return cipher.decrypt(encrypted).decode("utf-8")


# ------------------------
# Beispielaufruf
# ------------------------
if __name__ == "__main__":
    secret_text = "Mein geheimer API-Key: 12345-ABCDE"

    # Speichern
    secret_id = save_secret_to_db(secret_text)
    print(f"Secret gespeichert mit ID: {secret_id}")

    # Abrufen
    decrypted = get_secret_from_db(secret_id)
    print("Entschlüsselte Nachricht:", decrypted)


save_secret_to_db(secret) → verschlüsselt und speichert den Text in der DB

get_secret_from_db(entry_id) → liest die verschlüsselte Info aus der DB und entschlüsselt sie

Schlüssel (fernet.key) wird einmal erzeugt und danach wiederverwendet

Datenbank-Tabelle wird automatisch erstellt, falls sie nicht existiert